In [15]:
import torch
import numpy as np
import os
import sys

# Import model
from models.pgl import PGL
from utils.dataset import RecDataset
from utils.dataloader import TrainDataLoader

In [20]:
config = {
    # --- Data Paths ---
    'data_path': '../data/',  
    'dataset': 'baby',        
    'inter_file_name': 'baby.inter',
    
    # --- THÊM DÒNG NÀY ĐỂ SỬA LỖI ---
    'inter_splitting_label': 'x_label',  # <--- Quan trọng: RecDataset cần key này
    'field_separator': "\t",    
    
    # --- Model Params ---
    'model': 'PGL',
    'embedding_size': 64,
    'feat_embed_dim': 64,
    'n_mm_layers': 1,
    'n_ui_layers': 2,
    'knn_k': 10,
    'mm_image_weight': 0.1,
    'lambda_coeff': 0.9,
    'reg_weight': 0,
    'dropout': 0.2,
    'mode': 'local',
    
    # --- Dataset Specs ---
    'USER_ID_FIELD': 'userID',
    'ITEM_ID_FIELD': 'itemID',
    'TIME_FIELD': 'timestamp',
    'NEG_PREFIX': 'neg__',
    'train_batch_size': 2048,
    
    # Các tham số lọc data (thêm vào cho chắc chắn giống file yaml)
    'filter_out_cod_start_users': True, 
    
    'use_full_sampling': False,      # Lấy từ overall.yaml
    'use_neg_sampling': True,        # Lấy từ overall.yaml
    'use_neighborhood_loss': False,  # Thêm vào để tránh lỗi tiếp theo (PGL mặc định không dùng cái này)
    
    # --- Features ---
    'is_multimodal_model': True,
    'vision_feature_file': 'image_feat.npy',
    'text_feature_file': 'text_feat.npy',
    'end2end': False,
    
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}


In [27]:
MODEL_FILE_PATH = r'D:\Show_me_everything\Amazon-Recommend-System\saved_model\Dec-18-2025-11-42-46\best_model.pth'

In [33]:
def load_environment():
    """Load dataset và model"""
    print(f"--> Đang đọc dữ liệu từ: {config['data_path']}{config['dataset']}")
    
    # 1. Load Dataset
    # Model PGL cần dataset để dựng đồ thị (Graph) trong hàm __init__
    dataset = RecDataset(config) 
    if not hasattr(dataset, 'inter_num'):
        dataset.inter_num = len(dataset.df)
    train_data = TrainDataLoader(config, dataset, batch_size=config['train_batch_size'], shuffle=False)
    
    # 2. Init Model
    print("--> Khởi tạo model PGL...")
    model = PGL(config, train_data)
    
    # 3. Load Weights
    print(f"--> Load weights từ {MODEL_FILE_PATH}")
    if os.path.exists(MODEL_FILE_PATH):
        state_dict = torch.load(MODEL_FILE_PATH, map_location=config['device'])
        model.load_state_dict(state_dict)
        model.to(config['device'])
        model.eval()
    else:
        raise FileNotFoundError(f"Không tìm thấy file model: {MODEL_FILE_PATH}")
        
    return model, dataset, train_data

In [35]:
model, dataset, train_data = load_environment()

--> Đang đọc dữ liệu từ: ../data/baby
--> Khởi tạo model PGL...
--> Load weights từ D:\Show_me_everything\Amazon-Recommend-System\saved_model\Dec-18-2025-11-42-46\best_model.pth


In [29]:
def predict(model, user_internal_id, k=10):
    """
    Hàm infer không cần dùng DataLoader, tự tạo tensor input.
    """
    device = config['device']
    
    # Tạo tensor chứa ID user
    # Model PGL nhận input là tuple/list, và lấy phần tử đầu tiên làm user indices
    user_tensor = torch.tensor([user_internal_id]).to(device)
    dummy_interaction = (user_tensor, ) 
    
    with torch.no_grad():
        # Gọi hàm dự đoán của model
        # full_sort_predict trả về điểm số với TẤT CẢ item
        scores = model.full_sort_predict(dummy_interaction)
        
        # Flatten về 1 chiều vì chỉ có 1 user
        scores = scores.view(-1)
        
        # Lấy Top K item có điểm cao nhất
        top_scores, top_indices = torch.topk(scores, k)
        
    return top_indices.cpu().numpy(), top_scores.cpu().numpy()

In [36]:
def get_ground_truth(train_data, user_internal_id):
    """
    Lấy danh sách các Item mà User đã tương tác (từ dữ liệu nạp vào)
    """
    # TrainDataLoader đã tính sẵn cái này trong biến history_items_per_u
    # Nó là dict: {user_id: {item_id1, item_id2, ...}}
    if user_internal_id in train_data.history_items_per_u:
        return list(train_data.history_items_per_u[user_internal_id])
    return []

In [ ]:
test_user_id = 0 

ground_truth = get_ground_truth(train_data, test_user_id)
print(f"\n[GROUND TRUTH] User đã tương tác với {len(ground_truth)} items:")
print(f"IDs: {ground_truth}")
        
# Kiểm tra user hợp lệ
if test_user_id < dataset.get_user_num():
    print(f"\n--- Gợi ý cho User Internal ID: {test_user_id} ---")
    
    # 3. Chạy dự đoán
    items, scores = predict(model, test_user_id, k=5)
    
    for rank, (iid, score) in enumerate(zip(items, scores)):
        print(f"Top {rank+1}: Item ID {iid} (Score: {score:.4f})")
else:
    print("User ID không tồn tại.")


[GROUND TRUTH] User đã tương tác với 5 items:
IDs: [0, 1922, 1587, 1879, 3870]

--- Gợi ý cho User Internal ID: 0 ---
Top 1: Item ID 1587 (Score: 7.1996)
Top 2: Item ID 4841 (Score: 6.6478)
Top 3: Item ID 1543 (Score: 6.2638)
Top 4: Item ID 2322 (Score: 6.2615)
Top 5: Item ID 5455 (Score: 6.0126)

Lưu ý: Kết quả trả về là Internal Item ID.
